# 패션 이커머스 AI 어시스턴트 — 04. LLM 리뷰 요약

**03에서 확인한 문제**
- 리뷰를 통째로 섞은 v2는 문서가 104단어로 길어지면서 **상품 종류가 흐려졌다** ("작게 나오는 청바지"를 검색했는데 청바지가 0개)
- 직접 채점한 Precision@5도 v2(0.16)가 v1(0.28)보다 낮았다

**이번 노트북에서 할 일**
1. Gemini API로 상품별 리뷰를 **구조화된 JSON**으로 요약 (한 줄 요약, 장단점, 사이즈 핏, 소재, 계절, 상황)
2. 요약 품질 검증 — ① 사이즈 라벨을 규칙 기반 결과와 비교 ② LLM-as-judge로 **근거 없는 주장(환각)** 비율 측정
3. 요약으로 만든 검색 문서 **v3**로 03의 v1, v2와 다시 비교 — "리뷰를 정리해서 넣으면 상품 종류가 덜 흐려질까?"
4. 비용과 시간 측정 → 7만 개 전체로 늘리면 얼마나 들까?

**범위**: 비용과 시간을 아끼려고 03 검색어와 관련된 **8개 카테고리 × 25개 = 최대 200개 상품**만 요약해요.

## 0. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/fashion-ai-assistant'
PROCESSED_DIR = f'{PROJECT_DIR}/data/processed'
EMB_DIR = f'{PROJECT_DIR}/embeddings'
SUM_DIR = f'{PROJECT_DIR}/summaries'
os.makedirs(SUM_DIR, exist_ok=True)
K = 2

Mounted at /content/drive


In [ ]:
!pip install -q -U google-genai

import re
import json
import time
import enum
import numpy as np
import pandas as pd
from typing import List
from pydantic import BaseModel
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from IPython.display import display, HTML

pd.set_option('display.max_colwidth', 60)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 631.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 21.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


## 1. Gemini API 연결

**API 키는 코드에 직접 쓰지 마세요.** 노트북을 GitHub에 올리면 키가 그대로 노출돼요.

1. 코랩 왼쪽 사이드바의 **열쇠 아이콘(보안 비밀)** 클릭
2. **새 보안 비밀 추가** → 이름 `GEMINI_API_KEY`, 값에 API 키 입력
3. **노트북 액세스** 토글 켜기

모델은 비용이 낮은 Flash-Lite로 요약하고, 품질 검사(심사위원)는 한 단계 위인 Flash를 써요.
모델 이름 오류(404)가 나면 아래 셀의 목록에서 사용 가능한 이름으로 바꾸세요.

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

MODEL = 'gemini-3.5-flash-lite'   # 요약용 (저렴, 빠름)
JUDGE_MODEL = 'gemini-3.8-flash'  # 품질 검사용 (더 똑똑한 모델)

print('사용 가능한 Flash 계열 모델:')
print([m.name for m in client.models.list() if 'flash' in m.name][:15])

print('\n연결 테스트:', client.models.generate_content(model=MODEL, contents='한 단어로 인사해줘').text)

사용 가능한 Flash 계열 모델:
['models/gemini-2.5-flash', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-3-flash-preview', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3.1-flash-lite', 'models/gemini-3.1-flash-image-preview', 'models/gemini-3.1-flash-image', 'models/gemini-3.1-flash-lite-image', 'models/gemini-3.5-flash', 'models/gemini-3.5-flash-lite', 'models/gemini-omni-flash-preview']

연결 테스트: 안녕!


## 2. 데이터 불러오기
03과 똑같이 학습/평가를 나누고, **평가 정답 리뷰는 요약 재료에서 빼요** (데이터 누수 방지).

In [ ]:
items = pd.read_parquet(f'{EMB_DIR}/items_index.parquet')   # 03에서 저장 (임베딩과 같은 순서)
inter = pd.read_parquet(f'{PROCESSED_DIR}/interactions_k{K}.parquet')
reviews = pd.read_parquet(f'{PROCESSED_DIR}/reviews_all.parquet',
                          columns=['user_id', 'parent_asin', 'rating', 'text', 'helpful_vote'])

inter = inter.sort_values(['user_id', 'timestamp'])
test_df = inter[inter.groupby('user_id').cumcount(ascending=False) == 0]
test_keys = pd.MultiIndex.from_frame(test_df[['user_id', 'parent_asin']])

rev = reviews[reviews['parent_asin'].isin(set(items['parent_asin']))]
rev = rev[~pd.MultiIndex.from_frame(rev[['user_id', 'parent_asin']]).isin(test_keys)].copy()
rev['text'] = rev['text'].fillna('').str.replace(r'\s+', ' ', regex=True).str.strip()
rev = rev[rev['text'].str.len() >= 20]
print(f'상품 {len(items):,} | 요약 재료 리뷰 {len(rev):,}')

상품 70,641 | 요약 재료 리뷰 868,471


## 3. 요약할 상품 고르기

상품명으로 카테고리를 정해요. 이 카테고리는 **LLM과 무관한 기준**이라, 나중에 검색 결과의 상품 종류가 맞는지 자동으로 채점하는 데 써요.
리뷰가 15개 이상인 상품 중 리뷰가 많은 순서로 카테고리별 25개를 골라요.

In [ ]:
CATEGORIES = {   # 이름: (상품명 정규식, 제외할 정규식, 한국어)
    'watch':     (r'\bwatch(es)?\b',                        r'\b(band|strap|replacement|bands)\b', '시계'),
    'ring':      (r'\brings?\b',                            None, '반지'),
    'jeans':     (r'\bjeans?\b',                            None, '청바지'),
    'leggings':  (r'\bleggings?\b|\byoga pants?\b',         None, '레깅스'),
    'pajamas':   (r'\bpajamas?\b|\bpjs?\b|\bsleepwear\b',   None, '잠옷'),
    'crossbody': (r'\bcrossbody\b',                         None, '크로스백'),
    'blouse':    (r'\bblouses?\b',                          None, '블라우스'),
    'dress':     (r'\bdress(es)?\b',                        r'\bdress (shirt|shoes?|socks|pants)\b', '원피스'),
}
N_PER_CAT = 25
MIN_REVIEWS = 15

def title_category(title):
    t = str(title).lower()
    for cat, (pat, excl, _) in CATEGORIES.items():
        if re.search(pat, t) and not (excl and re.search(excl, t)):
            return cat
    return None

items['category'] = items['title'].map(title_category)
n_rev = rev.groupby('parent_asin').size()
items['n_reviews'] = items['parent_asin'].map(n_rev).fillna(0).astype(int)

selected = (items[items['category'].notna() & (items['n_reviews'] >= MIN_REVIEWS)]
            .sort_values('n_reviews', ascending=False)
            .groupby('category').head(N_PER_CAT)
            .reset_index(drop=True))
print(selected['category'].value_counts())
print(f'요약 대상: {len(selected)}개 상품')

# 선택한 상품의 리뷰만 미리 묶어 두기 (매번 전체 리뷰를 뒤지지 않도록)
rev_by_item = dict(tuple(rev[rev['parent_asin'].isin(set(selected['parent_asin']))].groupby('parent_asin')))

category
dress        25
leggings     25
crossbody    25
watch        25
blouse       25
ring         25
jeans        25
pajamas      25
Name: count, dtype: int64
요약 대상: 200개 상품


## 4. 프롬프트와 출력 형식

**구조화된 출력(Structured Output)**: 원하는 JSON 형식을 Pydantic 클래스로 정해 주면, 모델이 반드시 그 형식으로 답해요.
자유 형식 텍스트로 받으면 파싱이 깨지기 쉬워서, 서비스에서는 거의 항상 이 방식을 써요.

검색에 쓸 태그는 **영어**로, 사람이 읽을 요약은 **한국어**로 받아요.

In [ ]:
class SizeFit(str, enum.Enum):
    runs_small = 'runs_small'
    true_to_size = 'true_to_size'
    runs_large = 'runs_large'
    mixed = 'mixed'
    unknown = 'unknown'

class ReviewSummary(BaseModel):
    product_type: str          # 영어, 예: jeans, maxi dress, digital watch
    summary_ko: str            # 한 줄 요약
    pros_ko: List[str]         # 장점 최대 3개
    cons_ko: List[str]         # 단점 최대 3개
    size_fit: SizeFit
    size_evidence_ko: str      # 사이즈 판단 근거
    materials: List[str]       # 영어
    seasons: List[str]         # 영어: spring, summer, fall, winter
    occasions: List[str]       # 영어: office, wedding guest, workout ...
    style_tags: List[str]      # 영어

PROMPT = '''당신은 패션 쇼핑몰의 리뷰 분석가입니다. 아래 상품의 고객 리뷰를 읽고 정보를 정리하세요.

규칙:
- 리뷰에 실제로 나온 내용만 쓰세요. 리뷰에 없는 내용을 추측하지 마세요.
- 언급이 없으면 빈 리스트, 사이즈는 unknown으로 두세요.
- pros_ko, cons_ko는 여러 리뷰에서 반복된 내용 위주로 각각 최대 3개, 짧은 한국어 구로 쓰세요.
- size_fit: 작게 나온다는 리뷰가 대부분이면 runs_small, 크게 나온다면 runs_large, 정사이즈면 true_to_size, 의견이 갈리면 mixed.
- product_type, materials, seasons, occasions, style_tags는 검색에 쓰일 짧은 영어 단어로 쓰세요.

상품명: {title}

리뷰 ({n}개, [별점] 내용):
{reviews}'''

MAX_REVIEWS, MAX_CHARS = 40, 400

def pick_reviews(asin):
    r = rev_by_item[asin]
    helpful = r.sort_values('helpful_vote', ascending=False).head(30)
    low = r[(r['rating'] <= 3) & ~r.index.isin(helpful.index)].head(MAX_REVIEWS - len(helpful))  # 불만 리뷰도 꼭 포함
    return pd.concat([helpful, low])

def build_prompt(row):
    r = pick_reviews(row['parent_asin'])
    lines = [f"[{int(x.rating)}★] {x.text[:MAX_CHARS]}" for x in r.itertuples()]
    return PROMPT.format(title=row['title'], n=len(lines), reviews='\n'.join(lines))

print(build_prompt(selected.iloc[0])[:1500])

당신은 패션 쇼핑몰의 리뷰 분석가입니다. 아래 상품의 고객 리뷰를 읽고 정보를 정리하세요.

규칙:
- 리뷰에 실제로 나온 내용만 쓰세요. 리뷰에 없는 내용을 추측하지 마세요.
- 언급이 없으면 빈 리스트, 사이즈는 unknown으로 두세요.
- pros_ko, cons_ko는 여러 리뷰에서 반복된 내용 위주로 각각 최대 3개, 짧은 한국어 구로 쓰세요.
- size_fit: 작게 나온다는 리뷰가 대부분이면 runs_small, 크게 나온다면 runs_large, 정사이즈면 true_to_size, 의견이 갈리면 mixed.
- product_type, materials, seasons, occasions, style_tags는 검색에 쓰일 짧은 영어 단어로 쓰세요.

상품명: LILBETTER Women V-Neck Polka Dot Print Spaghetti Strap Boho Long Maxi Dresses

리뷰 (40개, [별점] 내용):
[5★] I bought this dress in a large and it fits perfectly. Normally i am a size 14/16 because i am top heavy. Its a very flattering dress. The only thing is, its a tad bit sheer. So you may need a slip or long camisole if ur worried about that. For people looking for a long dress, it is very long. I am 5'7 and it comes right above my ankles. It dies have a v in the back also so it does show a bit of ba
[5★] I love it! I thought it was a little to big until I realized there were strap adjustments! It came with a g

한 개만 먼저 돌려서 결과를 확인해요.

In [ ]:
class DailyQuotaExceeded(Exception):
    pass

def summarize(row, model=MODEL, retries=6):
    for attempt in range(retries):
        try:
            t0 = time.time()
            resp = client.models.generate_content(
                model=model,
                contents=build_prompt(row),
                config=types.GenerateContentConfig(
                    response_mime_type='application/json',
                    response_schema=ReviewSummary,
                    temperature=0.2,
                ),
            )
            parsed = resp.parsed or ReviewSummary.model_validate_json(resp.text)
            u = resp.usage_metadata
            return {'parent_asin': row['parent_asin'], **parsed.model_dump(mode='json'),
                    'input_tokens': u.prompt_token_count, 'output_tokens': u.candidates_token_count,
                    'seconds': round(time.time() - t0, 2)}
        except Exception as e:
            msg = str(e)
            if 'PerDay' in msg:                      # 하루 한도 → 기다려도 소용없음
                raise DailyQuotaExceeded(msg[:300])
            m = re.search(r"retryDelay['\"]?\s*:\s*['\"](\d+)", msg)   # 서버가 알려준 대기 시간
            wait = int(m.group(1)) + 2 if m else min(5 * 2 ** attempt, 90)
            print(f"[{row['parent_asin']}] {type(e).__name__} → {wait}s 후 재시도")
            time.sleep(wait)
    return None

## 5. 전체 요약 실행

- 결과를 한 줄씩 드라이브(`summaries.jsonl`)에 저장해요. **중간에 끊겨도 다시 실행하면 이어서** 진행돼요.
- `CONCURRENCY`는 동시에 보내는 요청 수예요. 무료 등급이라 요청 한도 오류가 계속 나면 1~2로 줄이세요.

In [ ]:
CACHE = f'{SUM_DIR}/summaries.jsonl'
REQUESTS_PER_MIN = 10   # 비율 제한 페이지의 RPM보다 조금 낮게 설정

done = set()
if os.path.exists(CACHE):
    with open(CACHE) as f:
        done = {json.loads(l)['parent_asin'] for l in f if l.strip()}
todo = selected[~selected['parent_asin'].isin(done)]
print(f'완료 {len(done)} | 남은 상품 {len(todo)}')

interval = 60 / REQUESTS_PER_MIN
t_start = time.time()
with open(CACHE, 'a') as f:
    for _, row in tqdm(todo.iterrows(), total=len(todo)):
        t = time.time()
        try:
            res = summarize(row)
        except DailyQuotaExceeded as e:
            print('\n오늘 사용량 한도에 도달했어요. 내일 이 셀을 다시 실행하면 이어서 진행돼요.')
            print(e)
            break
        if res:
            f.write(json.dumps(res, ensure_ascii=False) + '\n')
            f.flush()
        time.sleep(max(0, interval - (time.time() - t)))
print(f'소요 시간: {(time.time() - t_start) / 60:.1f}분')

완료 138 | 남은 상품 62


  0%|          | 0/62 [00:00<?, ?it/s]

[B01IB4FZPQ] ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please  → 3s 후 재시도
[B01IB4FZOC] ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please  → 3s 후 재시도
[B00GQNE27M] ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please  → 3s 후 재시도
[B097XMVWRL] ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please  → 3s 후 재시도
[B01JBVIXXO] ClientError → 10s 후 재시도
[B01IB4FZPQ] ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please  → 6s 후 재시도
[B01IB4FZOC] ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please  → 6s 후 재시도
[B00GQNE27M] ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please  → 6s 후 재

In [ ]:
with open(CACHE) as f:
    summ = pd.DataFrame([json.loads(l) for l in f if l.strip()]).drop_duplicates('parent_asin', keep='last')
summ = summ.merge(selected[['parent_asin', 'title', 'category', 'n_reviews', 'image_url']], on='parent_asin')
print(f'요약 완료: {len(summ)} / {len(selected)}')
summ[['category', 'title', 'summary_ko', 'size_fit']].head()

요약 완료: 200 / 200


,category,title,summary_ko,size_fit
0,watch,Accutime Kids LOL Surprise Turquoise Educational Learnin...,"어린이용 스마트 워치로 카메라, 게임, 만보기 등 다양한 기능이 있어 아이들이 좋아하지만, 쉽게 고장...",true_to_size
1,crossbody,Berchirly Small Vintage Canvas Leather Messenger Crossbo...,"작고 빈티지한 감성의 캔버스 크로스백으로, 외출 시 필수품을 가볍게 휴대하기 좋으나 일부 내구성 및 ...",true_to_size
2,dress,LILBETTER Women V-Neck Polka Dot Print Spaghetti Strap B...,"가벼우며 시원하고 편안한 핏의 보호 맥시 드레스로, 주머니가 있고 조절 가능한 스트랩이 있어 여름철에...",mixed
3,leggings,Aoxjox Women's High Waist Workout Gym Vital Seamless Leg...,체형을 잘 잡아주고 편안하지만 땀이 잘 보이거나 일부 색상에서 비침이 있다는 평이 있는 심리스 레깅스...,mixed
4,dress,Happy Sailed Women Summer Halter Neck Dress Floral Print...,체형을 드러내는 디자인과 조절 가능한 홀터넥 스트랩이 특징인 플로럴 프린트 비치 드레스입니다.,true_to_size


### 5-1. 요약 카드로 보기
실제 쇼핑몰의 "리뷰 한눈에 보기"처럼 보여줘요. 셀을 여러 번 실행하면 다른 상품이 나와요.

In [ ]:
SIZE_KO = {'runs_small': '작게 나와요', 'true_to_size': '정사이즈예요', 'runs_large': '크게 나와요',
           'mixed': '의견이 갈려요', 'unknown': '정보 없음'}

def summary_card(r):
    li = lambda xs: ''.join(f'<li>{x}</li>' for x in xs) or '<li>-</li>'
    img = f'<img src="{r.image_url}" style="height:140px;object-fit:contain">' if r.image_url else ''
    return f'''<div style="display:flex;gap:16px;border:1px solid #555;border-radius:8px;padding:12px;margin:8px 0;max-width:900px">
      <div style="width:150px;text-align:center">{img}</div>
      <div style="flex:1;font-size:13px">
        <div style="font-size:12px;opacity:.7">{r.category} · 리뷰 {r.n_reviews}개 · {r.product_type}</div>
        <div style="font-weight:600;margin:4px 0">{r.title[:90]}</div>
        <div style="margin:6px 0">💬 {r.summary_ko}</div>
        <div style="display:flex;gap:24px"><div>👍 장점<ul>{li(r.pros_ko)}</ul></div><div>👎 단점<ul>{li(r.cons_ko)}</ul></div></div>
        <div>📏 <b>{SIZE_KO.get(r.size_fit, r.size_fit)}</b> — {r.size_evidence_ko}</div>
        <div style="font-size:12px;opacity:.7">소재 {r.materials} · 계절 {r.seasons} · 상황 {r.occasions}</div>
      </div></div>'''

display(HTML(''.join(summary_card(r) for r in summ.sample(3).itertuples())))

## 6. 요약 품질 검증

LLM 요약은 그럴듯해 보여도 틀릴 수 있어요. 두 가지로 확인해요.

### 6-1. 사이즈 라벨 vs 규칙 기반
리뷰 전체에서 "runs small", "size up", "true to size" 같은 표현을 세서 규칙 기반 라벨을 만들고, LLM 라벨과 비교해요.
규칙 기반도 완벽하지 않으니 **"정답"이 아니라 "독립적인 두 번째 의견"** 으로 봐요.

In [ ]:
SMALL = r"runs? small|size up|sized up|order(?:ed)? (?:a|one) size (?:up|bigger|larger)|too small|too tight|smaller than expected"
LARGE = r"runs? (?:big|large)|size down|sized down|order(?:ed)? (?:a|one) size (?:down|smaller)|too big|too large|too loose|bigger than expected"
TRUE  = r"true to size|fits? perfect(?:ly)?|perfect fit|fits? as expected"

def rule_size(asin, min_mentions=3):
    t = rev_by_item[asin]['text'].str.lower()
    c = {'runs_small': t.str.contains(SMALL).sum(),
         'runs_large': t.str.contains(LARGE).sum(),
         'true_to_size': t.str.contains(TRUE).sum()}
    total = sum(c.values())
    if total < min_mentions:
        return 'unknown'
    label, n = max(c.items(), key=lambda x: x[1])
    return label if n / total >= 0.5 else 'mixed'

summ['size_rule'] = summ['parent_asin'].map(rule_size)
known = summ[summ['size_rule'] != 'unknown']
print(f"규칙 기반으로 판단 가능한 상품: {len(known)} / {len(summ)}")
print(f"LLM과 일치율: {(known['size_fit'] == known['size_rule']).mean():.1%}")
pd.crosstab(known['size_rule'], known['size_fit'], rownames=['규칙'], colnames=['LLM'])

규칙 기반으로 판단 가능한 상품: 171 / 200
LLM과 일치율: 43.9%


LLM,mixed,runs_large,runs_small,true_to_size,unknown
규칙,,,,,
mixed,18,5,7,1,0
runs_large,3,5,3,3,7
runs_small,8,1,43,5,8
true_to_size,30,4,10,9,1


In [ ]:
# 불일치 사례 직접 보기: 누가 맞았을까?
mismatch = known[known['size_fit'] != known['size_rule']]
mismatch[['category', 'title', 'size_rule', 'size_fit', 'size_evidence_ko']].head(10)

,category,title,size_rule,size_fit,size_evidence_ko
1,crossbody,Berchirly Small Vintage Canvas Leather Messenger Crossbo...,runs_small,true_to_size,치수가 설명된 것과 정확히 일치한다는 리뷰가 많습니다.
3,leggings,Aoxjox Women's High Waist Workout Gym Vital Seamless Leg...,true_to_size,mixed,"어떤 이에게는 정사이즈이거나 잘 맞지만, 어떤 이에게는 허리가 크거나 작게 느껴지는 등 의견이 갈립니다."
4,dress,Happy Sailed Women Summer Halter Neck Dress Floral Print...,mixed,true_to_size,사이즈가 대체로 잘 맞으며 정사이즈라는 의견이 많습니다.
6,dress,POPYOUNG Women's Long Sleeve T Shirt Dresses Casual Swin...,true_to_size,mixed,정사이즈라는 의견과 함께 작거나 크다는 의견이 골고루 존재하여 의견이 갈립니다.
7,dress,Jxstar Girls Summer Dress Sleeveless Printing Casual/Par...,true_to_size,mixed,"사이즈가 작거나 크다는 리뷰가 엇갈리며, 암홀이 너무 크거나 기장이 짧다는 의견 등 사이즈에 대한 의..."
9,watch,Invicta Pro Diver Stainless Steel Automatic Watch,runs_large,unknown,사이즈나 핏에 대한 구체적인 언급이 없습니다.
10,dress,Funny Party Hats Top Hats for Adults Dress Up Hats Costu...,runs_large,mixed,"머리 큰 사람에게 잘 맞거나 크다는 평과 함께, 어떤 이에게는 작거나 너무 커서 흘러내린다는 의견이 ..."
11,dress,TOPONSKY Women's Casual T-Shirt Long Sleeve Swing Dress,true_to_size,mixed,"대체로 잘 맞거나 정사이즈라는 의견이 많으나, 일부는 약간 작거나 혹은 헐렁하고 크다는 의견이 있어 ..."
15,crossbody,MiCoolker Small Crossbody Phone Purse Luxury Leather Tra...,true_to_size,mixed,"어떤 폰은 잘 맞고 넉넉하다고 한 반면, 폰 케이스를 끼우면 안 들어가거나 작다는 의견이 있어 의견이..."
17,crossbody,"NICGID Sling Bags, Chest Shoulder Backpacks Crossbody Pu...",mixed,runs_small,설명된 치수보다 실제 크기가 작아서 노트북이나 물건이 안 들어간다는 리뷰가 많습니다.


### 6-2. LLM-as-judge: 근거 없는 주장 찾기
요약의 장점, 단점, 사이즈 근거 문장 하나하나가 **실제 리뷰에 근거가 있는지** 다른 모델이 검사해요.
같은 모델이 자기 답을 채점하면 관대해지기 쉬워서 더 큰 모델(JUDGE_MODEL)을 써요. 비용 때문에 30개만 검사해요.

In [ ]:
class ClaimCheck(BaseModel):
    claim: str
    supported: bool
    reason_ko: str

class JudgeResult(BaseModel):
    checks: List[ClaimCheck]

JUDGE_PROMPT = '''아래 "주장" 각각이 "리뷰"에 근거가 있는지 판단하세요.
적어도 한 개 이상의 리뷰에 그 내용이 명시적으로 나오면 supported=true, 리뷰에 없거나 리뷰와 반대되면 false입니다.

주장:
{claims}

{reviews}'''

def judge(row):
    claims = row['pros_ko'] + row['cons_ko'] + ([row['size_evidence_ko']] if row['size_fit'] != 'unknown' else [])
    if not claims:
        return []
    reviews_part = build_prompt(row).split('리뷰 (', 1)[1]
    resp = client.models.generate_content(
        model=JUDGE_MODEL,
        contents=JUDGE_PROMPT.format(claims='\n'.join(f'- {c}' for c in claims), reviews='리뷰 (' + reviews_part),
        config=types.GenerateContentConfig(response_mime_type='application/json',
                                           response_schema=JudgeResult, temperature=0),
    )
    res = resp.parsed or JudgeResult.model_validate_json(resp.text)
    return [{'parent_asin': row['parent_asin'], 'category': row['category'], **c.model_dump()} for c in res.checks]

checks = []
for _, row in tqdm(summ.sample(min(30, len(summ)), random_state=0).iterrows(), total=min(30, len(summ))):
    try:
        checks += judge(row)
    except Exception as e:
        print('skip', row['parent_asin'], type(e).__name__)
checks = pd.DataFrame(checks)
print(f"검사한 주장 {len(checks)}개 중 근거 있음: {checks['supported'].mean():.1%}")
checks[~checks['supported']][['category', 'claim', 'reason_ko']].head(10)

  0%|          | 0/30 [00:00<?, ?it/s]

skip B0BVHDL9LM ServerError
skip B07R9SPDB9 ServerError
skip B07ZRZC5XV ServerError
skip B0B32185P1 ServerError
skip B07RLM8Q51 ServerError
skip B08PNNNP3R ClientError
skip B01G9O61LC ClientError
skip B01KF2QD9O ClientError
skip B019C4H93A ClientError
skip B01EN6KIVM ClientError
skip B07Q82VKL6 ClientError
skip B00ZET7UYQ ClientError
skip B07BTR665V ClientError
skip B07CN3D7DR ClientError
검사한 주장 109개 중 근거 있음: 97.2%


,category,claim,reason_ko
20,jeans,대다수의 리뷰에서 사이즈가 작게 나와서 한 사이즈 또는 두 사이즈 크게 주문해야 한다고 언급하고 있습니다.,전체 31개 리뷰 중 한 사이즈 또는 두 사이즈 크게 주문하라고 직접 언급한 리뷰는 약 3~4개에 불...
27,blouse,대다수의 리뷰어들이 사이즈가 작게 나와서 평소보다 2사이즈 이상 크게 주문해야 한다고 언급했습니다.,"일부 리뷰어가 2사이즈 크게 샀다는 내용을 언급하긴 했으나, 대다수의 리뷰어가 2사이즈 이상 크게 주..."
77,ring,가벼움,"반지가 가볍다는 언급은 없으며, 오히려 한 5점 리뷰에서는 예상보다 묵직하다('Heavier, not..."


## 7. 요약으로 검색 문서 v3 만들기

- **v1**: 상품명 + 특징 + 설명 (판매자 정보)
- **v2**: v1 + 리뷰 원문 3개 (03)
- **v3**: v1 + LLM이 정리한 속성 (상품 종류, 소재, 계절, 상황, 스타일, 사이즈 핏)

같은 200개 상품 안에서만 검색해서 **문서만 바꿨을 때의 차이**를 비교해요.

**자동 채점 (카테고리 정확도@5)**: 검색 결과 상위 5개 중 상품명 카테고리(3번에서 정한 것)가 검색 의도와 같은 비율.
8개 카테고리라 무작위로 뽑으면 12.5%예요.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
emb_model = SentenceTransformer('intfloat/multilingual-e5-base', device=device)
emb_model.max_seq_length = 256
if device == 'cuda':
    emb_model.half()

def embed(texts, prefix):
    return emb_model.encode([prefix + t for t in texts], batch_size=64, normalize_embeddings=True,
                            convert_to_numpy=True).astype(np.float32)

FIT_EN = {'runs_small': 'runs small, size up', 'runs_large': 'runs large, size down',
          'true_to_size': 'true to size', 'mixed': 'mixed sizing', 'unknown': ''}

def doc_v3(r):
    parts = [r.doc_v1,
             f"Type: {r.product_type}",
             f"Material: {', '.join(r.materials)}" if r.materials else '',
             f"Season: {', '.join(r.seasons)}" if r.seasons else '',
             f"Occasion: {', '.join(r.occasions)}" if r.occasions else '',
             f"Style: {', '.join(r.style_tags)}" if r.style_tags else '',
             f"Fit: {FIT_EN[r.size_fit]}" if FIT_EN[r.size_fit] else '']
    return ' | '.join(p for p in parts if p)

pool = summ.merge(items[['parent_asin', 'doc_v1']].reset_index(), on='parent_asin')   # index = 03 임베딩 행 번호
pool['doc_v3'] = [doc_v3(r) for r in pool.itertuples()]

pool_emb = {
    'v1': np.load(f'{EMB_DIR}/items_v1.npy').astype(np.float32)[pool['index'].values],
    'v2': np.load(f'{EMB_DIR}/items_v2.npy').astype(np.float32)[pool['index'].values],
    'v3': embed(pool['doc_v3'].tolist(), 'passage: '),
}
print('검색 풀:', len(pool), '| 예시 v3:', pool['doc_v3'].iloc[0][:300])

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

검색 풀: 200 | 예시 v3: Accutime Kids LOL Surprise Turquoise Educational Learning Touchscreen Smart Watch Toy for Girls, Boys, Toddlers - Selfie Cam, Learning Games, Alarm, Calculator, Pedometer & More (Model: LOL4320OMGAZ). The LOL Surprise kids smart watch comes loaded with kid-friendly features for your child to enjoy s


In [ ]:
QUERY_SET = [   # (검색어, 기대 카테고리)
    ('여름에 입기 좋은 시원한 린넨 원피스', 'dress'),
    ('결혼식 하객룩 원피스', 'dress'),
    ('출근할 때 입을 단정한 블라우스', 'blouse'),
    ('운동할 때 입는 하이웨이스트 레깅스', 'leggings'),
    ('실버 원석 반지', 'ring'),
    ('겨울에 입기 좋은 따뜻한 잠옷', 'pajamas'),
    ('black leather crossbody bag', 'crossbody'),
    ('방수되는 디지털 스포츠 시계', 'watch'),
    ('작게 나와서 한 치수 크게 사야 하는 청바지', 'jeans'),
    ('크게 나오는 편이라 한 치수 작게 사도 되는 원피스', 'dress'),
]

def pool_search(query, version, k=5):
    q = embed([query], 'query: ')[0]
    return np.argsort(-(pool_emb[version] @ q))[:k]

rows = []
for q, cat in QUERY_SET:
    row = {'query': q, 'expected': cat}
    for v in ['v1', 'v2', 'v3']:
        top = pool.iloc[pool_search(q, v)]
        row[v] = (top['category'] == cat).mean()
    rows.append(row)
cat_p5 = pd.DataFrame(rows).set_index('query')
print('카테고리 정확도@5')
display(cat_p5.round(2))
display(cat_p5[['v1', 'v2', 'v3']].mean().round(3).to_frame('평균').T)

카테고리 정확도@5


,expected,v1,v2,v3
query,,,,
여름에 입기 좋은 시원한 린넨 원피스,dress,0.2,0.2,0.2
결혼식 하객룩 원피스,dress,0.4,0.8,0.8
출근할 때 입을 단정한 블라우스,blouse,0.4,0.8,0.4
운동할 때 입는 하이웨이스트 레깅스,leggings,1.0,1.0,1.0
실버 원석 반지,ring,0.4,0.8,0.8
겨울에 입기 좋은 따뜻한 잠옷,pajamas,0.8,1.0,0.8
black leather crossbody bag,crossbody,1.0,1.0,1.0
방수되는 디지털 스포츠 시계,watch,1.0,1.0,1.0
작게 나와서 한 치수 크게 사야 하는 청바지,jeans,0.6,0.4,0.4


,v1,v2,v3
평균,0.58,0.7,0.66


### 7-1. 사이즈 조건 검색 확인
"작게 나오는 청바지" 검색 결과가 실제로 작게 나오는 상품인지, **규칙 기반 라벨**(LLM과 무관)로 확인해요.

In [ ]:
for q, cat in [QUERY_SET[-2], QUERY_SET[-1]]:
    print(f'\n🔎 {q}')
    out = {}
    for v in ['v1', 'v2', 'v3']:
        top = pool.iloc[pool_search(q, v)]
        out[v] = [f"[{c}|{s}] {t[:40]}" for c, s, t in zip(top['category'], top['size_rule'], top['title'])]
    display(pd.DataFrame(out, index=range(1, 6)))


🔎 작게 나와서 한 치수 크게 사야 하는 청바지


,v1,v2,v3
1,[jeans|runs_small] Chuanqi Women's High-waisted Ripped Hole,[jeans|true_to_size] Gloria Vanderbilt Women's Amanda Ba...,[jeans|mixed] Listenwind Fashion Toddler Baby Girl Clo
2,[jeans|true_to_size] Gloria Vanderbilt Women's Amanda Ba...,[blouse|mixed] SC88523A Renaissance Pirate White Chemis,[crossbody|runs_large] MINICAT Touch Screen Design Premi...
3,[crossbody|runs_large] MINICAT Touch Screen Design Premi...,[leggings|true_to_size] LMB | Seamless Full Length Leggi...,[jeans|runs_small] Chuanqi Women's High-waisted Ripped Hole
4,[jeans|true_to_size] June Julien Women's Basic Jeans Jeg...,[jeans|mixed] Listenwind Fashion Toddler Baby Girl Clo,[pajamas|runs_small] PNAEONG Women Pajama Set Print Shor...
5,[pajamas|runs_small] PNAEONG Women Pajama Set Print Shor...,[crossbody|runs_small] Viosi Small Leather Crossbody Bag...,"[crossbody|runs_small] AISPARKY Small Crossbody Purse,PU..."



🔎 크게 나오는 편이라 한 치수 작게 사도 되는 원피스


,v1,v2,v3
1,[pajamas|runs_small] BIG ELEPHANT Baby Boys' 1 Piece Lon...,[pajamas|runs_small] BIG ELEPHANT Baby Boys' 1 Piece Lon...,[ring|runs_small] BANG TI Titanium Quick Release Side Push
2,[pajamas|true_to_size] Battle Royale Boys' Gaming Pajama...,[crossbody|runs_small] Viosi Small Leather Crossbody Bag...,[pajamas|runs_small] BIG ELEPHANT Baby Boys' 1 Piece Lon...
3,[pajamas|true_to_size] Battle Royale Boys' Gaming Pajama...,[pajamas|runs_small] Battle Royale Boys' Gaming Pajamas ...,"[crossbody|runs_small] AISPARKY Small Crossbody Purse,PU..."
4,[crossbody|runs_small] Viosi Small Leather Crossbody Bag...,[blouse|runs_small] SheIn Women's Floral Printed V Neck ...,[dress|mixed] LILBETTER Women V-Neck Polka Dot Print S
5,[pajamas|runs_small] Battle Royale Boys' Gaming Pajamas ...,[ring|runs_small] BANG TI Titanium Quick Release Side Push,[jeans|mixed] Listenwind Fashion Toddler Baby Girl Clo


## 8. 비용과 시간
요약 1건당 토큰과 시간을 재서, 7만 개 전체로 늘렸을 때를 추정해요.
단가는 Gemini 가격 페이지에서 현재 값을 확인해서 넣으세요 (100만 토큰당 달러).

In [ ]:
PRICE_IN, PRICE_OUT = None, None   # 예: 0.10, 0.40  ← 가격 페이지에서 확인 후 입력

stats = summ[['input_tokens', 'output_tokens', 'seconds']].describe().loc[['mean', '50%', 'max']].round(1)
display(stats)

n_all = (items['n_reviews'] >= 3).sum()
tok_in, tok_out = summ['input_tokens'].mean() * n_all, summ['output_tokens'].mean() * n_all
print(f'리뷰 3개 이상인 상품 전체({n_all:,}개) 요약 시 토큰: 입력 {tok_in / 1e6:,.1f}M, 출력 {tok_out / 1e6:,.1f}M')
if PRICE_IN:
    print(f'예상 비용: ${tok_in / 1e6 * PRICE_IN + tok_out / 1e6 * PRICE_OUT:,.2f}')
print(f"동시 요청 {CONCURRENCY}개 기준 예상 시간: {summ['seconds'].mean() * n_all / CONCURRENCY / 3600:,.1f}시간")

,input_tokens,output_tokens,seconds
mean,2012.1,298.8,1.5
50%,2006.5,298.5,1.4
max,4019.0,381.0,2.7


리뷰 3개 이상인 상품 전체(53,468개) 요약 시 토큰: 입력 107.6M, 출력 16.0M
동시 요청 4개 기준 예상 시간: 5.5시간


## 9. 저장
5단계 챗봇에서 상품 질문에 답할 때 이 요약을 근거로 씁니다.

In [ ]:
summ.to_parquet(f'{SUM_DIR}/review_summaries.parquet', index=False)
pool[['parent_asin', 'doc_v3']].to_parquet(f'{SUM_DIR}/pool_docs.parquet', index=False)
np.save(f'{SUM_DIR}/pool_v3.npy', pool_emb['v3'].astype(np.float16))
if len(checks):
    checks.to_csv(f'{SUM_DIR}/judge_checks.csv', index=False)
print(sorted(os.listdir(SUM_DIR)))

['judge_checks.csv', 'pool_docs.parquet', 'pool_v3.npy', 'review_summaries.parquet', 'summaries.jsonl']


In [ ]:
PRICE_IN, PRICE_OUT = 0.30, 2.50          # gemini-3.5-flash-lite, 100만 토큰당 달러
BATCH_IN, BATCH_OUT = 0.15, 1.25          # Batch API 가격

print('=== 0. 요약 개수 ===')
print(f'저장된 요약: {len(summ)} / {len(selected)}')
print(summ['category'].value_counts().to_string())

print('\n=== 1. 요약 예시 (카테고리별 1개) ===')
for r in summ.groupby('category').head(1).itertuples():
    print(f'\n[{r.category}] {r.title[:70]} (리뷰 {r.n_reviews}개)')
    print(f'  요약: {r.summary_ko}')
    print(f'  장점: {r.pros_ko} | 단점: {r.cons_ko}')
    print(f'  사이즈: {r.size_fit} (규칙: {r.size_rule}) — {r.size_evidence_ko}')

print('\n=== 2. 사이즈 라벨 비교 ===')
known = summ[summ['size_rule'] != 'unknown']
print(f'규칙으로 판단 가능: {len(known)} / {len(summ)} | 일치율: {(known.size_fit == known.size_rule).mean():.1%}')
print(pd.crosstab(known['size_rule'], known['size_fit'], rownames=['규칙'], colnames=['LLM']).to_string())
print('\n불일치 사례:')
for r in known[known.size_fit != known.size_rule].head(8).itertuples():
    t = rev_by_item[r.parent_asin]['text'].str.lower()
    cnt = {k: int(t.str.contains(p).sum()) for k, p in [('small', SMALL), ('large', LARGE), ('true', TRUE)]}
    print(f'- [{r.category}] {r.title[:50]} | 규칙 {r.size_rule} {cnt} | LLM {r.size_fit}: {r.size_evidence_ko}')

print('\n=== 3. LLM-as-judge ===')
if len(checks):
    print(f"검사한 주장 {len(checks)}개 | 근거 있음 {checks['supported'].mean():.1%}")
    print(checks.groupby('category')['supported'].agg(['mean', 'count']).round(2).to_string())
    print('\n근거 없음 판정:')
    for r in checks[~checks['supported']].itertuples():
        print(f'- [{r.category}] "{r.claim}" → {r.reason_ko}')

print('\n=== 4. 카테고리 정확도@5 ===')
print(cat_p5.round(2).to_string())
print(cat_p5[['v1', 'v2', 'v3']].mean().round(3).to_string())
for q, cat in [QUERY_SET[-2], QUERY_SET[-1]]:
    print(f'\n🔎 {q}')
    for v in ['v1', 'v2', 'v3']:
        top = pool.iloc[pool_search(q, v)]
        print(f'  {v}: ' + ' / '.join(f"[{c}|{s}]" for c, s in zip(top['category'], top['size_rule'])))

print('\n=== 5. 비용과 시간 ===')
print(summ[['input_tokens', 'output_tokens', 'seconds']].describe().loc[['mean', '50%', 'max']].round(1).to_string())
n_all = int((items['n_reviews'] >= 3).sum())
tin, tout = summ['input_tokens'].mean() * n_all, summ['output_tokens'].mean() * n_all
print(f'리뷰 3개 이상 상품 {n_all:,}개 전체: 입력 {tin/1e6:,.1f}M, 출력 {tout/1e6:,.1f}M 토큰')
print(f'예상 비용: 일반 ${tin/1e6*PRICE_IN + tout/1e6*PRICE_OUT:,.2f} | 배치 ${tin/1e6*BATCH_IN + tout/1e6*BATCH_OUT:,.2f}')
print(f"200개 실제 처리 비용: ${summ['input_tokens'].sum()/1e6*PRICE_IN + summ['output_tokens'].sum()/1e6*PRICE_OUT:.3f}")

=== 0. 요약 개수 ===
저장된 요약: 200 / 200
category
watch        25
crossbody    25
dress        25
leggings     25
blouse       25
ring         25
jeans        25
pajamas      25

=== 1. 요약 예시 (카테고리별 1개) ===

[watch] Accutime Kids LOL Surprise Turquoise Educational Learning Touchscreen  (리뷰 729개)
  요약: 어린이용 스마트 워치로 카메라, 게임, 만보기 등 다양한 기능이 있어 아이들이 좋아하지만, 쉽게 고장나거나 과열되는 등의 내구성 문제와 품질 불량이 보고되었습니다.
  장점: ['다양한 게임과 카메라 기능', '아이들이 좋아하는 디자인', '시간 개념 학습에 도움'] | 단점: ['금방 고장나거나 켜지지 않음', '과열 및 고주파 소음 발생', '방수가 되지 않음']
  사이즈: true_to_size (규칙: unknown) — 스트랩 사이즈가 잘 맞고 가벼워서 아이들 손목에 부담이 없다는 리뷰가 있습니다.

[crossbody] Berchirly Small Vintage Canvas Leather Messenger Crossbody Bag Pack Or (리뷰 785개)
  요약: 작고 빈티지한 감성의 캔버스 크로스백으로, 외출 시 필수품을 가볍게 휴대하기 좋으나 일부 내구성 및 수납 입구 크기에 대한 불만이 있습니다.
  장점: ['가볍고 수납이 편리함', '빈티지하고 멋스러운 디자인', '튼튼한 캔버스 소재'] | 단점: ['벨크로 마감 및 내구성 부족', '입구가 좁아 물건 꺼내기가 불편함', '지퍼 고장 문제']
  사이즈: true_to_size (규칙: runs_small) — 치수가 설명된 것과 정확히 일치한다는 리뷰가 많습니다.

[dress] LILBETTER Women V-Neck Polka Dot Print Spag

## 10. 정리해볼 질문 (블로그 / 포트폴리오용)

1. 요약 카드를 직접 읽어보니 어땠나? 잘된 요약과 틀린 요약 사례 하나씩
2. 사이즈 라벨은 규칙 기반과 몇 % 일치했나? 불일치 사례를 보면 누가 맞았나? (규칙이 틀린 경우: "not too small" 같은 부정문)
3. LLM-as-judge 결과 근거 있는 주장의 비율은? 근거 없는 주장(환각)은 어떤 유형이었나?
4. 카테고리 정확도@5에서 v3는 v1, v2보다 나았나? → 03의 "리뷰를 통째로 넣으면 상품 종류가 흐려진다"는 가설 검증
5. 7만 개 전체를 요약하면 비용과 시간은? 실제 서비스라면 언제 요약을 다시 만들까? (새 리뷰가 N개 쌓일 때만 등)

1. 요약 카드를 직접 읽어보니 어땠나?

요약 200개가 모두 정해진 JSON 형식으로 저장됐고, 형식 오류는 없었다.

잘된 사례: Pyroluna 새틴 슬립(잠옷)은 "부드럽고 신축성 있지만 작게 나와서 한두 치수 크게 주문해야 하고, 레이스 색상이 사진과 다르다"로 요약됐다. 장점, 단점, 사이즈 정보가 구매 결정에 필요한 내용 위주로 잘 정리됐고, 규칙 기반 사이즈 라벨(runs_small)과도 일치했다. LLM이 상품명 키워드보다 실제 상품을 더 정확히 파악한 경우도 있었다. AuntTaylor 상품은 상품명에 “blouse”가 들어가 블라우스로 분류됐지만, LLM은 리뷰를 읽고 "오픈 프론트 가디건"이라고 정확히 요약했다.

틀린 사례: 같은 AuntTaylor 상품의 사이즈 근거는 "대다수의 리뷰어들이 평소보다 2사이즈 이상 크게 주문해야 한다고 언급했다"였다. 6-2 검사에서 이 문장은 근거 없음으로 판정됐다. 실제로는 일부 리뷰만 2사이즈 크게 샀다고 했고, 정사이즈라거나 크게 샀더니 너무 컸다는 리뷰도 있었다. 작게 나온다는 방향은 맞았지만 "대다수", “2사이즈 이상”으로 정도를 과장한 것이다.

2. 사이즈 라벨은 규칙 기반과 몇 % 일치했나? 누가 맞았나?

규칙 기반으로 판단할 수 있는 상품 171개 중 LLM과 라벨이 일치한 비율은 43.9%로 낮았다. 가장 많이 엇갈린 경우는 규칙이 "정사이즈"라고 했는데 LLM이 "의견이 갈림(mixed)"이라고 한 경우로, 30개였다.

불일치 사례를 보면 어느 한쪽이 일방적으로 맞았다기보다, 원인이 양쪽에 있었다.

LLM 쪽 원인: 리뷰를 고른 방식의 편향. 규칙은 상품의 리뷰 전체(수백 개)를 셌지만, LLM은 리뷰 40개만 봤다. 게다가 불만 리뷰를 일부러 포함하도록 골랐기 때문에, 사이즈 불만이 실제보다 많이 보였다. 예를 들어 POPYOUNG 원피스는 전체 리뷰에서 정사이즈 언급이 91회, 작다 34회, 크다 17회로 정사이즈가 64%였는데, LLM은 mixed로 판단했다. 6-2에서 근거 없음으로 판정된 3건 중 2건이 "대다수가 작다고 했다"는 과장이었던 것도 같은 원인으로 보인다.

규칙 쪽 원인: 문맥을 모른다. Berchirly 크로스백은 규칙이 "작게 나옴"으로 판단했지만, 실제로는 "가방 용량이 작다"는 리뷰였다. 가방에는 옷처럼 사이즈 핏이라는 개념이 맞지 않는다. Invicta 시계처럼 사이즈 언급이 7개뿐인데 규칙이 크게 나온다고 판단한 경우도 있었다. “not too small” 같은 부정문도 규칙은 구분하지 못한다.

개선 방향은 역할을 나누는 것이다. 사이즈 언급 개수처럼 셀 수 있는 것은 코드로 리뷰 전체에서 세고, LLM에는 그 숫자와 함께 대표 리뷰를 넘겨 해석만 맡긴다. 리뷰를 고를 때도 불만 리뷰를 일부러 늘리지 않고 실제 별점 비율대로 고른다.

3. LLM-as-judge 결과 근거 있는 주장의 비율은?

무작위 30개 상품의 요약에서 주장 109개를 더 큰 모델(gemini-3.8-flash)로 검사한 결과, 97.2%(106개)가 리뷰에 근거가 있었다. 근거 없음으로 판정된 3건은 두 유형이었다.

정도의 과장 (2건): "대다수의 리뷰에서 한두 사이즈 크게 주문하라고 했다." 실제로는 31개 리뷰 중 3~4개만 그렇게 말했다. 방향은 맞지만 "대다수"라는 표현이 틀렸다.
사실과 반대 (1건): 반지 상품의 장점 "가벼움". 리뷰에는 가볍다는 언급이 없었고, 오히려 "생각보다 묵직하다"는 리뷰가 있었다.

없는 내용을 지어내는 경우보다 일부 리뷰의 내용을 전체 의견처럼 일반화하는 경우가 더 많았다. 쇼핑몰에서 "대부분 작다고 해요"라고 잘못 보여주면 고객이 불필요하게 큰 사이즈를 사서 반품으로 이어질 수 있어서, 이런 과장도 실제 서비스에서는 중요한 오류다.

한계도 있다. 검사한 상품이 30개뿐이고, 심사 모델도 LLM이라 판정이 틀릴 수 있다. 또 심사 모델은 요약에 쓴 리뷰 40개만 보고 판단했기 때문에, 그 40개 자체가 치우쳐 있으면(2번) 잡아내지 못한다.

4. 카테고리 정확도@5에서 v3는 v1, v2보다 나았나?

같은 200개 상품 안에서 검색어 10개로 비교한 결과, 카테고리 정확도@5는 v1 0.58, v2 0.70, v3 0.66이었다. 무작위로 고르면 0.125다.

03에서 세운 "리뷰를 통째로 넣으면 상품 종류가 흐려진다"는 가설은 이번 실험에서는 확인되지 않았다. 오히려 리뷰 원문을 넣은 v2가 가장 좋았고, LLM이 정리한 속성을 넣은 v3는 v2보다 조금 낮았다. "결혼식 하객룩 원피스"(0.4 → 0.8)나 "실버 원석 반지"(0.4 → 0.8)처럼 리뷰에 사용 상황과 소재가 나오는 검색어에서 v2와 v3가 크게 좋아졌다. 반면 "작게 나오는 청바지"에서는 03과 마찬가지로 v1(0.6)이 v2, v3(0.4)보다 나았다.

03과 결과가 다른 이유는 실험 조건이 달랐기 때문으로 보인다. 03은 상품명이 비어 있는 상품까지 포함한 7만 개 전체에서, 검색 의도에 맞는지를 사람이 채점했다. 04는 리뷰가 15개 이상인 상품 200개 안에서 상품 종류만 자동으로 채점했다. 리뷰가 많은 상품은 대표 리뷰의 질이 좋아서, 리뷰가 상품 정보를 흐리기보다 보완했을 가능성이 크다. 다만 v2와 v3의 차이(0.04)는 검색 결과 50개 중 2개 차이라, v2가 확실히 낫다고 결론 내리기는 어렵다.

자동 채점 기준에도 한계가 있었다. 상품명 키워드로 카테고리를 정했기 때문에 열쇠고리(key ring)가 반지로, 파티 모자(dress up hats)가 원피스로, 가디건이 블라우스로 분류된 경우가 있었다.

가장 중요한 발견: 사이즈 방향을 구분하지 못한다

이 내용은 4번 답에 이어서 쓰세요.

사이즈 조건 검색은 세 방법 모두 실패했다. "크게 나오는 편이라 한 치수 작게 사도 되는 원피스"를 검색하면 정반대인 "작게 나오는" 상품들이 상위에 나왔다. v3 문서에는 “Fit: runs small”처럼 사이즈 정보가 명시돼 있었는데도 마찬가지였다.

임베딩은 "작게 나온다"와 "크게 나온다"를 둘 다 "사이즈 이야기"라는 비슷한 의미로 보기 때문에, 방향을 구분하지 못한다. 이런 조건은 임베딩 유사도가 아니라 필터로 처리해야 한다. 검색어에서 "원피스(상품 종류) + 크게 나옴(사이즈 조건)"을 뽑아낸 뒤, size_fit이 runs_large인 원피스만 남기고 그 안에서 의미 검색을 하는 방식이다. 4단계에서 만든 구조화된 요약(size_fit, product_type)이 이 필터의 재료가 된다.

5. 7만 개 전체를 요약하면 비용과 시간은? 언제 요약을 다시 만들까?

요약 1건당 평균 입력 2,012토큰, 출력 299토큰, 1.5초가 걸렸고, 200개를 요약하는 데 약 $0.27이 들었다. 리뷰가 3개 이상인 상품 53,468개 전체로 늘리면 입력 1억 760만 토큰, 출력 1,600만 토큰으로 약 $72, Batch API를 쓰면 약 $36로 추정된다.

출력 토큰은 입력의 7분의 1인데, 단가가 8배 비싸서 비용은 출력이 더 컸다(1건당 입력 약 $0.0006, 출력 약 $0.00075). 비용을 줄이려면 리뷰를 덜 넣는 것보다 출력을 줄이는 게 효과적이다. 예를 들어 사람이 읽는 한국어 요약은 상품 페이지에 실제로 보여줄 상품만 만들고, 검색용 영어 태그만 전체 상품에 만드는 식이다.

시간은 요청을 하나씩 보내면 약 22시간이다. 실제로는 무료 등급에서 4개를 동시에 보내자 15초 만에 요청 한도(429)에 걸렸다. 그래서 요청 속도 조절, 서버가 알려준 대기 시간만큼 기다렸다가 재시도, 완료한 상품을 저장해 중단돼도 이어서 하는 처리를 추가했다. 대량 처리는 실시간 API보다 Batch API가 비용도 절반이고 한도 문제도 적다.

실제 서비스라면 리뷰가 달릴 때마다 요약을 새로 만들 필요는 없다. 새 리뷰가 일정 개수(예: 20개) 쌓이거나 기존 리뷰 수의 20% 이상 늘었을 때만 다시 요약하고, 이 작업은 하루 한 번 Batch API로 모아서 처리한다. 신상품은 리뷰가 최소 개수(예: 5개)에 도달했을 때 처음 요약한다.
